# WFCR — Visualização de uma solução

Este notebook lê um *trial* salvo em um banco SQLite (produzido por `wfcr run` ou `wfcr run-all`),
reconstrói o problema a partir dos arquivos `.turb`/`.cable` e desenha a solução com matplotlib.

Use os seletores para escolher o **banco**, o **algoritmo**, a **instância** e se quer
desenhar o **melhor** trial ou o **mediano**. Ajuste as pastas abaixo se necessário.


In [1]:
from pathlib import Path

# Onde procurar os bancos SQLite (glob "*.db").
DB_DIRS = [Path("..") / "databases", Path("..")]

# Pastas onde procurar os arquivos .turb/.cable das instâncias.
INSTANCE_DIRS = [Path("..") / "instances", Path("..") / "instances_mid"]

# Onde salvar os PNGs (usado apenas se o checkbox "Salvar PNG" estiver marcado).
IMAGES_DIR = Path("..") / "images"

DB_DIRS, INSTANCE_DIRS

([PosixPath('../databases'), PosixPath('..')],
 [PosixPath('../instances'), PosixPath('../instances_mid')])

In [2]:
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

COLORS = ["green", "dodgerblue", "purple", "darkorange", "firebrick", "deeppink"]


def has_flow_schema(path) -> bool:
    """Só bancos com a coluna `flow` em `edges` permitem reconstruir os cabos."""
    try:
        conn = sqlite3.connect(path)
        try:
            cols = [row[1] for row in conn.execute("PRAGMA table_info(edges)")]
        finally:
            conn.close()
        return "flow" in cols
    except sqlite3.Error:
        return False


def find_instance(instance: str) -> Path:
    for d in INSTANCE_DIRS:
        p = Path(d) / f"{instance}.turb"
        if p.exists():
            return p
    raise FileNotFoundError(f"Instância '{instance}' não encontrada em {INSTANCE_DIRS}")


def load_problem(instance: str):
    turb = find_instance(instance)

    nodes = []
    for line in turb.read_text().splitlines():
        if not line.strip():
            continue
        x, y, w = line.split()
        nodes.append((float(x), float(y), int(w)))

    cables = []
    for line in turb.with_suffix(".cable").read_text().splitlines():
        if not line.strip():
            continue
        parts = line.split()
        cables.append((int(parts[0]), float(parts[1])))
    cables.sort(key=lambda c: c[0])

    return nodes, cables


def load_trial(conn: sqlite3.Connection, trial_id: int):
    trial = pd.read_sql_query(
        "SELECT id, algorithm, instance, total_cost, duration_ms FROM trials WHERE id = ?",
        conn,
        params=[trial_id],
    ).iloc[0]
    edges = pd.read_sql_query(
        "SELECT u, v, flow FROM edges WHERE trial_id = ? ORDER BY u",
        conn,
        params=[trial_id],
    )
    parents = dict(zip(edges["u"], edges["v"]))
    flows = dict(zip(edges["u"], edges["flow"]))
    return trial, parents, flows

In [3]:
def pick_trial(conn, instance, algorithm, mode="best"):
    df = pd.read_sql_query(
        "SELECT id, total_cost FROM trials WHERE instance = ? AND algorithm = ?",
        conn,
        params=[instance, algorithm],
    )
    if df.empty:
        return None
    if mode == "median":
        target = df["total_cost"].median()
        row = df.iloc[(df["total_cost"] - target).abs().argmin()]
    else:
        row = df.loc[df["total_cost"].idxmin()]
    return int(row["id"])


def cable_for_flow(flow, cables):
    for power, _price in cables:
        if flow <= power:
            return power
    return 127


def build_figure(nodes, cables, parents, flows, cost, algorithm):
    xs = np.array([n[0] for n in nodes])
    ys = np.array([n[1] for n in nodes])
    substation = next(i for i, n in enumerate(nodes) if n[2] < 0)

    powers = sorted(
        {cable_for_flow(flows[x], cables) for x in range(len(nodes)) if x != substation}
    )

    fig, ax = plt.subplots(figsize=(10, 8))

    for x in range(len(nodes)):
        if x == substation:
            continue
        p = parents[x]
        idx = powers.index(cable_for_flow(flows[x], cables))
        ax.annotate(
            "",
            xy=(xs[p], ys[p]),
            xytext=(xs[x], ys[x]),
            arrowprops={
                "arrowstyle": "-|>",
                "color": COLORS[idx % len(COLORS)],
                "lw": 1 + idx * 1.1,
                "shrinkA": 5,
                "shrinkB": 5,
            },
            zorder=2,
        )

    for i, (x, y, w) in enumerate(nodes):
        if w < 0:
            ax.scatter(x, y, marker="s", s=50, color="red", zorder=5)
        else:
            ax.scatter(x, y, marker="o", s=10, facecolors="white", edgecolors="black", zorder=5)

    ax.set_aspect("equal")
    ax.axis("off")

    handles = [Line2D([], [], linestyle="none") for _ in range(2)]
    labels = [f"Total cost: R$ {cost:,.2f}", f"Algorithm: {algorithm}"]
    for idx, power in enumerate(powers):
        handles.append(Line2D([], [], color=COLORS[idx % len(COLORS)], lw=4, marker="o", markersize=8))
        labels.append(f"cable cap: {power}")
    ax.legend(handles, labels, loc="lower left", frameon=True, fontsize=9)

    fig.tight_layout()
    return fig

In [4]:
import ipywidgets as widgets
from IPython.display import display

DB_FILES = [p for d in DB_DIRS for p in sorted(d.glob("*.db")) if has_flow_schema(p)]

db_w = widgets.Dropdown(
    options=[str(p) for p in DB_FILES],
    value=str(DB_FILES[0]) if DB_FILES else None,
    description="Banco",
    layout=widgets.Layout(width="55%"),
)
algo_w = widgets.Dropdown(description="Algoritmo", layout=widgets.Layout(width="55%"))
inst_w = widgets.Dropdown(description="Instância", layout=widgets.Layout(width="55%"))
mode_w = widgets.ToggleButtons(options=["best", "median"], description="Trial")
save_w = widgets.Checkbox(value=False, description="Salvar PNG")
out = widgets.Output()

conn = None
_updating = False


def open_db(path):
    global conn
    if conn is not None:
        conn.close()
    conn = sqlite3.connect(path)


def instances_for(algorithm):
    return pd.read_sql_query(
        "SELECT DISTINCT instance FROM trials WHERE algorithm = ? ORDER BY instance",
        conn,
        params=[algorithm],
    )["instance"].tolist()


def set_algorithms():
    global _updating
    _updating = True
    algos = pd.read_sql_query(
        "SELECT DISTINCT algorithm FROM trials ORDER BY algorithm", conn
    )["algorithm"].tolist()
    algo_w.options = algos
    algo_w.value = algos[0] if algos else None
    _updating = False


def set_instances():
    global _updating
    _updating = True
    options = instances_for(algo_w.value)
    inst_w.options = options
    inst_w.value = options[0] if options else None
    _updating = False


def render(*_):
    if _updating or conn is None or not algo_w.value or not inst_w.value:
        return
    out.clear_output()
    with out:
        trial_id = pick_trial(conn, inst_w.value, algo_w.value, mode_w.value)
        if trial_id is None:
            print("Nenhum trial para esse par (algoritmo, instância).")
            return
        trial, parents, flows = load_trial(conn, trial_id)
        nodes, cables = load_problem(trial["instance"])
        fig = build_figure(nodes, cables, parents, flows, trial["total_cost"], trial["algorithm"])
        plt.show()
        print(
            f"trial #{trial_id} | {trial['algorithm']} | {trial['instance']} | "
            f"R$ {trial['total_cost']:,.2f} | {trial['duration_ms']} ms"
        )
        if save_w.value:
            IMAGES_DIR.mkdir(parents=True, exist_ok=True)
            path = IMAGES_DIR / f"{trial['instance']}-{trial['algorithm']}.png"
            fig.savefig(path, dpi=150, bbox_inches="tight")
            print(f"PNG salvo em {path}")
        plt.close(fig)


def on_db_change(*_):
    open_db(db_w.value)
    set_algorithms()
    set_instances()
    render()


def on_algo_change(*_):
    set_instances()
    render()


db_w.observe(on_db_change, names="value")
algo_w.observe(on_algo_change, names="value")
inst_w.observe(render, names="value")
mode_w.observe(render, names="value")
save_w.observe(render, names="value")

display(widgets.VBox([db_w, algo_w, inst_w, widgets.HBox([mode_w, save_w]), out]))
on_db_change()